In [37]:
import sys, json, warnings
from pathlib import Path
import pandas as pd, numpy as np

warnings.filterwarnings("ignore")
sys.path.append(str(Path.cwd().parent / "src"))
from config import DATA_PROCESSED, RESULTS_TABLES, RANDOM_SEED
from features import (
    add_lexical_features,
    add_evidentiary_features,
    add_sentiment_features,
    add_time_of_day,
)

np.random.seed(RANDOM_SEED)
print("Ready")

Ready


In [38]:
reddit  = pd.read_parquet(DATA_PROCESSED / "reddit_processed.parquet")
twitter = pd.read_parquet(DATA_PROCESSED / "twitter_processed.parquet")
print("Reddit:", reddit.shape, " Twitter:", twitter.shape)

Reddit: (9406, 35)  Twitter: (11724, 27)


In [39]:
reddit  = add_lexical_features(reddit,  "clean_text")
twitter = add_lexical_features(twitter, "clean_text")
print("Lexical done")

Lexical done


In [42]:
# Evidentiary — links need the RAW text, lexicon uses cleaned text
import importlib
import features.evidentiary_features as evidentiary_module

evidentiary_module = importlib.reload(evidentiary_module)
add_evidentiary_features = evidentiary_module.add_evidentiary_features

reddit  = add_evidentiary_features(reddit, raw_col="self_text", text_col="clean_text")
twitter = add_evidentiary_features(twitter, raw_col="tweet_text", text_col="clean_text")

print("Evidentiary done")
print("  Reddit  rows with link_count > 0:", (reddit["link_count"] > 0).sum())
print("  Twitter rows with link_count > 0:", (twitter["link_count"] > 0).sum())
print("  Reddit  mean rigor_score:", round(reddit["rigor_score"].mean(), 4))
print("  Twitter mean rigor_score:", round(twitter["rigor_score"].mean(), 4))

Evidentiary done
  Reddit  rows with link_count > 0: 492
  Twitter rows with link_count > 0: 5263
  Reddit  mean rigor_score: 0.0444
  Twitter mean rigor_score: 0.1989


In [43]:
#  If your dataset is very large, sample first for testing:
reddit  = reddit.sample(5000, random_state=RANDOM_SEED)
twitter = twitter.sample(5000, random_state=RANDOM_SEED)

import time

started = time.perf_counter()
print(f"Starting Reddit sentiment/emotion features ({len(reddit):,} rows)...", flush=True)
reddit = add_sentiment_features(reddit, "clean_text")
print(f"Reddit complete after {time.perf_counter() - started:.1f}s", flush=True)

print(f"Starting X sentiment/emotion features ({len(twitter):,} rows)...", flush=True)
twitter = add_sentiment_features(twitter, "clean_text")
print(f"X complete after {time.perf_counter() - started:.1f}s", flush=True)
print("Sentiment done")

Starting Reddit sentiment/emotion features (5,000 rows)...
Reddit complete after 763.7s
Starting X sentiment/emotion features (5,000 rows)...
X complete after 1529.7s
Sentiment done


In [44]:
reddit  = add_time_of_day(reddit)
twitter = add_time_of_day(twitter)
print("Temporal done")

Temporal done


In [45]:
reddit["platform"]  = "Reddit"
twitter["platform"] = "X"

reddit.to_parquet(DATA_PROCESSED / "reddit_with_features.parquet", index=False)
twitter.to_parquet(DATA_PROCESSED / "twitter_with_features.parquet", index=False)

combined = pd.concat([reddit, twitter], ignore_index=True)
combined.to_parquet(DATA_PROCESSED / "combined_with_features.parquet", index=False)

print("Saved. Reddit:", len(reddit), "Twitter:", len(twitter), "Combined:", len(combined))

Saved. Reddit: 5000 Twitter: 5000 Combined: 10000


In [46]:
key = ["fk_grade", "ttr", "rigor_score", "sentiment_score",
       "evidentiary_count", "link_count", "engagement_normalized"]
print("Reddit means:\n",  reddit[key].mean(numeric_only=True).round(4))
print("\nX means:\n",     twitter[key].mean(numeric_only=True).round(4))

Reddit means:
 fk_grade                 8.4914
ttr                      0.8636
rigor_score              0.0481
sentiment_score          0.0000
evidentiary_count        0.9330
link_count               0.1012
engagement_normalized    0.2278
dtype: float64

X means:
 fk_grade                 10.8937
ttr                       0.8806
rigor_score               0.1982
sentiment_score           0.0000
evidentiary_count         0.5488
link_count                0.4956
engagement_normalized     0.4170
dtype: float64
